# BASICS-CDSS Quick Start Tutorial

**Duration:** 10 minutes  
**Level:** Beginner  
**Goal:** Get started with BASICS-CDSS evaluation framework

## What is BASICS-CDSS?

**BASICS-CDSS** (*Beyond Accuracy: Simulation-based Integrated Critical-Safety evaluation for Clinical Decision Support Systems*) is a framework for evaluating clinical decision support systems based on **behavioral safety under uncertainty** rather than just predictive accuracy.

### Key Questions BASICS-CDSS Answers:
1. **Calibration:** Does the system know when it's uncertain?
2. **Coverage-Risk:** Can the system safely abstain from risky predictions?
3. **Harm-Aware:** Does the system minimize harm in high-risk scenarios?
4. **Robustness:** How does the system behave under information missingness, ambiguity, or conflict?

This tutorial demonstrates a complete evaluation workflow in under 50 lines of code.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for development
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import BASICS-CDSS modules
from basics_cdss.scenario import instantiate_scenarios
from basics_cdss.metrics import (
    expected_calibration_error,
    selective_prediction_metrics,
    compute_harm_metrics
)
from basics_cdss.visualization import (
    plot_reliability_diagram,
    plot_coverage_risk_curve,
    plot_harm_by_tier
)

print("✓ BASICS-CDSS imported successfully")
print(f"  NumPy version: {np.__version__}")
print(f"  Pandas version: {pd.__version__}")

## 2. Create Sample Scenarios

For this quick start, we'll create synthetic scenarios directly (without loading SynDX archetypes).

In [ ]:
# Create sample archetypes (normally these would come from SynDX)
np.random.seed(42)

n_samples = 100

# Simulate a CDSS triage task
sample_data = pd.DataFrame({
    'patient_id': range(n_samples),
    'heart_rate': np.random.randint(60, 140, n_samples),
    'temperature': np.random.uniform(36.0, 39.5, n_samples),
    'blood_pressure_sys': np.random.randint(90, 180, n_samples),
    'respiratory_rate': np.random.randint(12, 30, n_samples),
})

# True risk tier (ground truth)
risk_tiers = []
for idx, row in sample_data.iterrows():
    # Simple rule-based ground truth
    if row['temperature'] > 38.5 or row['heart_rate'] > 120 or row['blood_pressure_sys'] > 160:
        risk_tiers.append('high')
    elif row['temperature'] > 37.5 or row['heart_rate'] > 100:
        risk_tiers.append('medium')
    else:
        risk_tiers.append('low')

sample_data['risk_tier'] = risk_tiers

# True labels (1 = requires intervention)
sample_data['y_true'] = (sample_data['risk_tier'] == 'high').astype(int)

print(f"Created {n_samples} sample scenarios")
print(f"\nRisk tier distribution:")
print(sample_data['risk_tier'].value_counts())
print(f"\nIntervention rate: {sample_data['y_true'].mean():.1%}")

## 3. Simulate a Mock CDSS Model

We'll simulate a CDSS model that makes predictions with varying confidence levels. A well-calibrated model should have confidence scores that match its actual accuracy.

In [ ]:
# Simulate CDSS predictions
# Model A: Poorly calibrated (overconfident)
np.random.seed(123)

y_true = sample_data['y_true'].values
risk_tier_array = sample_data['risk_tier'].values

# Generate predicted probabilities (somewhat correlated with truth)
base_probs = y_true * 0.7 + (1 - y_true) * 0.3 + np.random.normal(0, 0.15, n_samples)
base_probs = np.clip(base_probs, 0.05, 0.95)

# Make model overconfident (push probabilities toward extremes)
y_prob = np.where(base_probs > 0.5, 
                  base_probs + 0.2, 
                  base_probs - 0.1)
y_prob = np.clip(y_prob, 0.01, 0.99)

# Predicted labels
y_pred = (y_prob > 0.5).astype(int)

# Calculate basic accuracy
accuracy = (y_pred == y_true).mean()

print(f"Model Accuracy: {accuracy:.1%}")
print(f"Positive Prediction Rate: {y_pred.mean():.1%}")
print(f"\nExample predictions:")
print(pd.DataFrame({
    'True Label': y_true[:5],
    'Predicted Prob': y_prob[:5].round(3),
    'Predicted Label': y_pred[:5],
    'Risk Tier': risk_tier_array[:5]
}))

## 4. Evaluate Beyond Accuracy: Calibration

**Calibration** asks: *Does the model know when it's uncertain?*

A well-calibrated model's confidence scores should match its accuracy. For example, when the model predicts 80% confidence, it should be correct 80% of the time.

In [ ]:
# Compute Expected Calibration Error (ECE)
ece = expected_calibration_error(y_true, y_prob, n_bins=10)

print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"  → Lower is better (perfect calibration = 0)")
print(f"  → This model's ECE of {ece:.4f} suggests {'poor' if ece > 0.1 else 'good'} calibration")

# Visualize calibration
from basics_cdss.metrics.calibration import reliability_curve

confs, accs, counts = reliability_curve(y_true, y_prob, n_bins=10)

fig, ax = plot_reliability_diagram(confs, accs, counts, 
                                     title="CDSS Calibration: Confidence vs Accuracy")
plt.tight_layout()
plt.show()

print("\n✓ Calibration analysis complete")

## 5. Evaluate Coverage-Risk Tradeoff

**Coverage-Risk** asks: *Can the model safely abstain from risky predictions?*

A CDSS should be able to identify cases where it's uncertain and defer to human clinicians. The coverage-risk curve shows this tradeoff: as we allow the model to abstain from low-confidence predictions, the error rate on remaining predictions should decrease.

In [ ]:
# Compute selective prediction metrics
sp_metrics = selective_prediction_metrics(y_true, y_prob)

print(f"Area Under Risk-Coverage Curve (AURC): {sp_metrics.aurc:.4f}")
print(f"  → Lower is better (perfect selective prediction ≈ 0)")
print(f"  → Measures how well the model can identify uncertain cases")

# Visualize coverage-risk tradeoff
fig, ax = plot_coverage_risk_curve(
    sp_metrics.coverage_curve, 
    sp_metrics.risk_curve,
    title="Coverage-Risk Tradeoff: Can the CDSS Safely Abstain?"
)
plt.tight_layout()
plt.show()

print("\n✓ Coverage-risk analysis complete")

## 6. Evaluate Harm-Aware Performance

**Harm-Aware** asks: *Does the model minimize harm in high-risk scenarios?*

Not all errors are equal. Missing a high-risk patient (false negative) is typically more harmful than over-triaging a low-risk patient (false positive). Harm-aware metrics weight errors by their clinical severity.

In [ ]:
# Compute harm-aware metrics
# Default harm weights: high=10x, medium=3x, low=1x
harm_metrics = compute_harm_metrics(y_true, y_pred, risk_tier_array)

print(f"Weighted Harm Loss: {harm_metrics.weighted_harm_loss:.4f}")
print(f"  → Lower is better")
print(f"  → Weights errors by risk tier (high=10x, medium=3x, low=1x)")
print(f"\nHarm by Risk Tier:")
for tier, loss in harm_metrics.harm_by_tier.items():
    print(f"  {tier:>8}: {loss:.4f}")

print(f"\nEscalation Failures: {harm_metrics.escalation_failures}")
print(f"  → High-risk cases missed by the model")

# Visualize harm by tier
fig, ax = plot_harm_by_tier(harm_metrics.harm_by_tier, 
                             title="Harm-Aware Loss by Risk Tier")
plt.tight_layout()
plt.show()

print("\n✓ Harm-aware analysis complete")

## 7. Summary

You've completed your first BASICS-CDSS evaluation! Here's what we learned:

### Beyond Accuracy Metrics:

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Accuracy** | See above | Traditional metric (insufficient for safety-critical systems) |
| **ECE** | See above | Calibration: Does the model know when it's uncertain? |
| **AURC** | See above | Selective prediction: Can the model safely abstain? |
| **Weighted Harm** | See above | Clinical impact: Does the model minimize harm? |

### Key Takeaways:

1. **Accuracy alone is insufficient** for evaluating clinical decision support systems
2. **Calibration** ensures the model's confidence matches its actual performance
3. **Coverage-Risk** enables safe abstention when the model is uncertain
4. **Harm-aware metrics** prioritize clinical safety over raw performance

### Next Steps:

- **[Notebook 01](01_basics_scenario_instantiation.ipynb)**: Deep-dive into scenario generation with perturbations
- **[Notebook 02](02_basics_beyond_accuracy_metrics.ipynb)**: Comprehensive calibration analysis
- **[Notebook 03](03_coverage_risk_selective_prediction.ipynb)**: Advanced coverage-risk strategies
- **[Notebook 04](04_harm_aware_evaluation.ipynb)**: Custom harm weights and escalation analysis
- **[Notebook 05](05_end_to_end_pipeline.ipynb)**: Complete evaluation workflow